# SearchLite-RL headroom study: RESUME after the step-89 crash
Branch `experiment/headroom-grpo` (needs the `trajectory_signature` fix pushed). Finished stages (headroom eval, signal probe, completed runs) are skipped; the crashed run is set aside automatically and retrained (~12 min). Run cells in order. Exit code 3 = a gate stopped the study: paste the printed decision, do not continue.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -euo pipefail
cd /content
if [ -d Pilot-project ]; then cd Pilot-project && git fetch -q && git checkout -q experiment/headroom-grpo && git pull -q; else git clone -q -b experiment/headroom-grpo https://github.com/aasritha-oql7872/Pilot-project.git; fi
cd /content/Pilot-project
python -c "import trl,peft,transformers" 2>/dev/null && echo "dependencies already installed" || { pip install -q -r requirements.txt; pip uninstall -y -q torchvision torchaudio torchao; echo "INSTALLED: restart the session (Runtime > Restart session), then continue with the next cell"; }
git log --oneline -3

In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
git log --oneline -1
grep -q "_query_of" src/train_grpo.py && echo "fix present" || { echo "FIX MISSING: push it, then re-run the setup cell"; exit 1; }


In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
D=/content/drive/MyDrive/SearchLite-RL
mkdir -p outputs $D/study_sft $D/study_hard
for s in study_sft study_hard; do ln -sfn $D/$s outputs/$s; done
mkdir -p outputs/main_v2
[ -f outputs/main_v2/adapter/adapter_config.json ] || cp -rn $D/outputs/main_v2/adapter outputs/main_v2/ 2>/dev/null || cp -rn $D/main_v2/adapter outputs/main_v2/ 2>/dev/null || echo "condition A adapter not on Drive (M1 would be retrained by run_sft_study.sh)"
nvidia-smi -L
ls outputs/study_sft | head -40
python -m pytest -q 2>&1 | tail -3

In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
echo "--- gate decisions so far ---"
cat outputs/study_hard/decision_headroom.json 2>/dev/null || echo "(headroom gate not run yet)"
cat outputs/study_hard/decision_signal.json 2>/dev/null || echo "(signal gate not run yet)"
ls outputs/study_hard


In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
# PHASE 3-5: headroom check -> signal check -> matched GRPO runs -> hard_dev tables. Gated (exit 3 = stop and report).
bash run_headroom_study.sh 2>&1 | tee -a outputs/study_hard/run_$(date +%Y%m%d_%H%M).log | tail -80
cat outputs/study_hard/summary_hard_dev.md

In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
# Look at the evidence before choosing the headline model (hard_dev only), then FREEZE it in the next cell.
O=outputs/study_hard
cat $O/decision_headroom.json $O/decision_signal.json
cat $O/summary_dev_regression.md $O/summary_twohop_dev_regression.md
python - <<'PY'
import json
d=json.load(open("outputs/study_hard/traj_stats.json"))
for k,v in d.items(): print(k, "informative groups:",v["all_groups_nonzero_std"],"twohop second-search:",v["twohop"]["second_search"],"bonus-created variance:",v["twohop"]["bonus_created_variance"])
PY